In [4]:
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

In [7]:
# Load and clean CSV files
X_test_df = pd.read_csv("test_dataset.csv").apply(pd.to_numeric, errors='coerce')
y_test_df = pd.read_csv("test_labels.csv").apply(pd.to_numeric, errors='coerce')

# Convert to PyTorch tensors
X_test = torch.tensor(X_test_df.values, dtype=torch.float32)
y_test = torch.tensor(y_test_df.values.squeeze(), dtype=torch.float32)

print("Data loaded successfully! Shape of features:", X_test.shape)

Data loaded successfully! Shape of features: torch.Size([1000, 12])


In [12]:
from model import MythicThreatClassifier # Imports architecture from model.py

model_files = {
    "ARES": "models/model_ares.pth",
    "ATHENA": "models/model_athena.pth",
    "APOLLO": "models/model_apollo.pth"
}

results = {}

for name, path in model_files.items():
    model = OracleModel()
    model.load_state_dict(torch.load(path))
    model.eval()
    
    with torch.no_grad():
        outputs = model(X_test).squeeze()
        
        # Apply sigmoid if outputs are raw logits
        probs = torch.sigmoid(outputs) if (outputs.min() < 0 or outputs.max() > 1) else outputs
        
        # Decision threshold at 0.50[cite: 2]
        preds = (probs >= 0.50).int()
        y_true = y_test.int()
        
        # Compute metrics
        acc = accuracy_score(y_true, preds)
        recall = recall_score(y_true, preds, zero_division=0)
        cm = confusion_matrix(y_true, preds)
        tn, fp, fn, tp = cm.ravel()
        
        results[name] = {"accuracy": acc, "recall": recall, "fn": fn}
        print(f"Model: {name} | Accuracy: {acc:.4f} | Recall: {recall:.4f} | False Negatives: {fn}")

ImportError: cannot import name 'MythicThreatClassifier' from 'model' (/Users/mariana/MADMINDS/MADMINDS/Europe/Greece/model.py)